# Sujet 2 — Pyronear : Parsing et manipulation des données

Construit la table "une ligne par séquence" décrite dans la méthode suggérée :
- parsing du nom de dossier (site, azimut, date)
- lecture des labels YOLO (score max, score moyen, nb de frames positives)
- durée de la séquence (à partir des timestamps des images)
- jointure avec `sites.csv` (lat/lon)
- récupération du FWI du jour via l'API risque (avec cache local)

In [1]:
from pathlib import Path
import re
import json
import time
from datetime import date
import pandas as pd
import numpy as np

root = Path("mines_pyronear_data") #donne le chemin d'accès du dossier
root.exists()

True

## 1. Parsing du nom de séquence

Récupération du format : `<site>_<azimut>_<timestamp>`.

On vérifie que le timestamp matche le format ISO attendu (`YYYY-MM-DDTHH-MM-SS`),
pour détecter d'éventuels noms de dossier mal formés.

In [ ]:
TEST_ts = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}-\d{2}-\d{2}$") #créer un moule de date au bon format

def parse_seq(name: str):

    #on sépare en 3 parties : site_azimut_timestamp
    parts = name.rsplit("_", 2)

    site, azimut, ts = parts

    if not TEST_ts.match(ts):
        raise ValueError(f"Timestamp mal formé dans {name!r}: {ts!r}")

    date = ts[:10]

    return site, date, azimut

# essai sur le premier nom de séquence du dossier
premier_test = next((root / "wildfire").iterdir())
print(premier_test, "->", parse_seq(premier_test.name))


mines_pyronear_data\wildfire\pyronear-force-06_cabanelle_125_2024-02-16T12-11-03 -> ('pyronear-force-06_cabanelle', '2024-02-16', '125')


## 2. Lecture des labels YOLO + agrégation par séquence

Format de chaque ligne d'un fichier label : `classe x_centre y_centre largeur hauteur score`
(classe 0 = fumée).

Pour chaque séquence, on calcule :
- `score_max` : score de confiance maximum sur toutes les frames
- `score_mean` : score moyen **parmi les frames avec détection** (les frames sans détection ne sont pas comptées comme score 0 dans la moyenne, sinon on dilue artificiellement le signal)
- `n_frames` : nombre total de frames dans la séquence

In [3]:
def parse_label_file(txt_path: Path):
    # Retourne la liste des scores sans espace et renvoie du vide s'il n'y a rien
    score = 0
    text = txt_path.read_text().strip()
    if not text:
        return score

    fields = text.split()
    if len(fields) < 6:
        print(f"ligne malformée dans {txt_path}: {text!r}")
        return score
    
    score = float(fields[5])
    return score


def seq_features(seq_dir: Path):
    # Récupère les fichiers text et les trie par date
    label_files = sorted((seq_dir / "labels").glob("*.txt"))

    all_scores = []

    for txt in label_files:
        score = parse_label_file(txt)
        all_scores.append(score)

    score_max = max(all_scores, default=0.0)
    score_mean = float(np.mean(all_scores)) if all_scores else 0.0
    n_frames = len(label_files)

    return dict(
        score_max=score_max,
        score_mean=score_mean,
        n_frames=n_frames,
    )

# sanity check
print(seq_features(premier_test))

{'score_max': 0.875, 'score_mean': 0.8135462857142858, 'n_frames': 7}


## 3. Construction de la table complète

On parcourt `wildfire/` et `fp/`, on parse chaque nom de dossier, on calcule les features,
et on enrichit avec lat/lon via `sites.csv`.

In [26]:
sites_df = pd.read_csv(root / "sites.csv").set_index("site")

rows = []
errors = []

for label in ("wildfire", "fp","camera_name"):

    label_dir = root / label #adaptation mac/windows

    if not label_dir.exists():
        print(f"dossier manquant: {label_dir}")
        continue

    for seq_dir in sorted(label_dir.iterdir()):
        if not seq_dir.is_dir():
            continue
        try:
            site, date, azimut = parse_seq(seq_dir.name)
        except ValueError as e:
            errors.append((seq_dir.name, str(e)))
            continue

        if site not in sites_df.index:
            errors.append((seq_dir.name, f"site {site!r} introuvable dans sites.csv"))
            continue

        lat, lon = sites_df.loc[site, ["lat", "lon"]]
        feats = seq_features(seq_dir)

        #création de toutes les colonnes
        rows.append(dict(
            seq=seq_dir.name,
            label=label,
            site=site,
            azimut=azimut,
            date=date,
            lat=lat,
            lon=lon,
            **feats,
        ))

df_1 = pd.DataFrame(rows)

print(f"{len(df_1)} séquences chargées, {len(errors)} erreurs/ignorées.")

df_1.head()

dossier manquant: mines_pyronear_data\camera_name
236 séquences chargées, 0 erreurs/ignorées.


,seq,label,site,azimut,date,lat,lon,score_max,score_mean,n_frames
0,pyronear-force-06_cabanelle_125_2024-02-16T12-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-16,43.786647,7.419608,0.875000,0.813546,7
1,pyronear-force-06_cabanelle_125_2024-02-24T08-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-24,43.786647,7.419608,0.695801,0.557687,7
2,pyronear-force-06_cabanelle_125_2024-02-27T13-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-27,43.786647,7.419608,0.823242,0.684522,10
3,pyronear-force-06_cabanelle_244_2024-02-23T10-...,wildfire,pyronear-force-06_cabanelle,244,2024-02-23,43.786647,7.419608,0.798828,0.682324,10
4,pyronear-force-06_cabanelle_244_2024-09-14T17-...,wildfire,pyronear-force-06_cabanelle,244,2024-09-14,43.786647,7.419608,0.573242,0.438599,4


In [27]:
print(pd.read_csv("mines_pyronear_data_2/sequences_info.csv", nrows=0).columns.tolist())

df_2 = pd.read_csv("mines_pyronear_data_2/sequences_info.csv")
vraies_colonnes = ["sequence_id", "label", "camera_name", "camera_azimuth","last_seen_at", "lat", "lon", "max_conf"]
df_2 = df_2[vraies_colonnes]
dictionnaire_noms = {
    "sequence_id": "seq",
    "label": "label",
    "camera_name": "site",
    "camera_azimuth": "azimut",
    "last_seen_at": "date",
    "lat": "lat",
    "lon": "lon",
    "max_conf": "score_max"
}
df_2 = df_2.rename(columns=dictionnaire_noms)

df_2.head(5)

['sequence_id', 'label', 'sublabel', 'camera_id', 'camera_name', 'organization_id', 'lat', 'lon', 'elevation', 'camera_azimuth', 'cone_angle', 'angle_of_view', 'alert_time', 'last_seen_at', 'detections_count', 'max_conf', 'is_wildfire', 'is_validated']


,seq,label,site,azimut,date,lat,lon,score_max
0,40766,fp,moret-sur-loing-02,359.0,2026-05-08T05:08:49.031245,48.379167,2.820833,0.467
1,40828,fp,croix-augas-01,285.0,2026-05-08T14:21:14.046482,48.426746,2.710876,0.435
2,40864,fp,croix-augas-01,327.0,2026-05-08T15:47:21.436881,48.426746,2.710876,0.314
3,41016,fp,nemours-02,291.0,2026-05-09T19:52:00.165332,48.260483,2.706383,0.566
4,41047,fp,nemours-02,291.0,2026-05-10T09:06:55.664288,48.260483,2.706383,0.340


In [28]:
df = pd.concat([df_1, df_2], axis=0, ignore_index=True)
df.head(250)

,seq,label,site,azimut,date,lat,lon,score_max,score_mean,n_frames
0,pyronear-force-06_cabanelle_125_2024-02-16T12-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-16,43.786647,7.419608,0.875000,0.813546,7.0
1,pyronear-force-06_cabanelle_125_2024-02-24T08-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-24,43.786647,7.419608,0.695801,0.557687,7.0
2,pyronear-force-06_cabanelle_125_2024-02-27T13-...,wildfire,pyronear-force-06_cabanelle,125,2024-02-27,43.786647,7.419608,0.823242,0.684522,10.0
3,pyronear-force-06_cabanelle_244_2024-02-23T10-...,wildfire,pyronear-force-06_cabanelle,244,2024-02-23,43.786647,7.419608,0.798828,0.682324,10.0
4,pyronear-force-06_cabanelle_244_2024-09-14T17-...,wildfire,pyronear-force-06_cabanelle,244,2024-09-14,43.786647,7.419608,0.573242,0.438599,4.0
...,...,...,...,...,...,...,...,...,...,...
245,40445,fp,nemours-01,194.0,2026-05-05T15:51:51.691572,48.260483,2.706383,0.641000,NaN,NaN
246,40472,fp,nemours-01,52.0,2026-05-05T17:09:01.190813,48.260483,2.706383,0.651000,NaN,NaN
247,40673,fp,moret-sur-loing-02,359.0,2026-05-07T05:14:17.604640,48.379167,2.820833,0.649000,NaN,NaN
248,40743,fp,croix-augas-01,285.0,2026-05-07T17:56:43.344492,48.426746,2.710876,0.440000,NaN,NaN


## 4. Jointure avec le score de risque

On cache récupère les fwi de chaque destination en prenant en entrée la longitude, la lattitude, la date. On met ensuite les données sur le tableau.

In [29]:
import requests
import io
from PIL import Image

EFFIS_WMS = "https://maps.effis.emergency.copernicus.eu/effis"
LAYER = "mf010.fwi"

# EFFIS European FWI danger classes
FWI_CLASSES = [
    (5.2, "very_low"),
    (11.2, "low"),
    (21.3, "moderate"),
    (38.0, "high"),
    (50.0, "very_high"),
    (float("inf"), "extreme"),
]

def fwi_class(value: float) -> str:
    for threshold, label in FWI_CLASSES:
        if value < threshold:
            return label
    return "extreme"

def get_fwi(lat : float, lon : float, target_date : date, layer : str = LAYER):
    half = 0.5
    grid = 11
    params = {
        "service": "WMS",
        "version": "1.1.1",
        "request": "GetMap",
        "layers": layer,
        "styles": "",
        "srs": "EPSG:4326",
        "bbox": f"{lon - half},{lat - half},{lon + half},{lat + half}",
        "width": grid,
        "height": grid,
        "format": "image/tiff",
        "time": target_date,
    }

    r = requests.get(EFFIS_WMS, params=params, timeout=30)
    r.raise_for_status()
    return r

fwi_results = []
fwi_errors = []

for i, row in df.iterrows():
    try:
        r = get_fwi(row["lat"], row["lon"], row["date"])

        img = np.array(Image.open(io.BytesIO(r.content)))

        fwi_val = float(img[5,5])
        
        res = {
            "fwi": fwi_val,
            "fwi_class": fwi_class(fwi_val)
        }
    except Exception as e:
        
        fwi_errors.append((row["seq"], str(e)))
        res = {"fwi": None, "fwi_class": None}
        
    fwi_results.append(res)


df["fwi"] = [r.get("fwi") for r in fwi_results]
df["fwi_class"] = [r.get("fwi_class") for r in fwi_results]

if fwi_errors:
    print(f"\n⚠ {len(fwi_errors)} erreurs de traitement/API (lignes avec fwi=None):")
    for seq, msg in fwi_errors[:10]:
        print(" -", seq, "->", msg)

In [30]:

masque = (df["fwi"] != 0)
dfmasque = df[masque]
dfmasque.head(250)

,seq,label,site,azimut,date,lat,lon,score_max,score_mean,n_frames,fwi,fwi_class
10,pyronear-force-06_courmettes_080_2024-09-07T11...,wildfire,pyronear-force-06_courmettes,080,2024-09-07,43.715592,7.026486,0.785645,0.660889,6.0,2.034760,very_low
49,pyronear-sdis-07_brison_039_2025-05-11T09-39-59,wildfire,pyronear-sdis-07_brison,039,2025-05-11,44.545155,4.216534,0.182617,0.091309,2.0,2.994781,very_low
57,pyronear-sdis-07_brison_110_2024-09-05T11-05-34,wildfire,pyronear-sdis-07_brison,110,2024-09-05,44.545155,4.216534,0.707031,0.482459,10.0,0.582558,very_low
61,pyronear-sdis-07_brison_127_2025-05-13T10-21-54,wildfire,pyronear-sdis-07_brison,127,2025-05-13,44.545155,4.216534,0.164429,0.054810,3.0,0.597397,very_low
65,pyronear-sdis-07_brison_226_2025-05-29T15-25-39,wildfire,pyronear-sdis-07_brison,226,2025-05-29,44.545155,4.216534,0.550781,0.513509,3.0,30.315994,high
...,...,...,...,...,...,...,...,...,...,...,...,...
424,41220,fp,nemours-01,147.0,2026-05-10T17:10:46.281760,48.260483,2.706383,0.452000,NaN,NaN,6.782845,low
425,41221,fp,croix-augas-01,60.0,2026-05-10T17:11:14.156580,48.426746,2.710876,0.323000,NaN,NaN,5.922340,low
426,41222,fp,moret-sur-loing-01,235.0,2026-05-10T17:12:40.956893,48.379167,2.820833,0.480000,NaN,NaN,6.848351,low
427,41223,fp,croix-augas-01,60.0,2026-05-10T17:22:44.211427,48.426746,2.710876,0.515000,NaN,NaN,5.922340,low


## 5. Vérifications rapides et export

- valeurs manquantes (notamment `fwi`/`fwi_class` si des appels API ont échoué)
- distribution des classes de risque par label (feu / FP)
- export de la table en CSV pour réutilisation

In [7]:
print("Valeurs manquantes par colonne:")
print(df.isna().sum())

print()
print("Répartition label x fwi_class:")
print(pd.crosstab(df["label"], df["fwi_class"]))

print()
print("Stats score_max par label:")
print(df.groupby("label")["score_max"].describe())

OUT_PATH = Path("sequences_with_fwi.csv")
df.to_csv(OUT_PATH, index=False)
print(f"\nTable exportée -> {OUT_PATH.resolve()}")

Valeurs manquantes par colonne:
seq           0
label         0
site          0
azimut        0
date          0
lat           0
lon           0
score_max     0
score_mean    0
n_frames      0
fwi           0
fwi_class     0
dtype: int64

Répartition label x fwi_class:
fwi_class  high  low  moderate  very_high  very_low
label                                              
fp            4    5         6          1       100
wildfire      3    1        10          0       106

Stats score_max par label:
          count      mean       std       min       25%       50%       75%  \
label                                                                         
fp        116.0  0.520563  0.195358  0.000000  0.370525  0.521350  0.683500   
wildfire  120.0  0.707907  0.128941  0.164429  0.646240  0.735108  0.792602   

               max  
label               
fp        0.890600  
wildfire  0.893555  

Table exportée -> C:\Users\octav\cours-info\HACK-PYRON\sequences_with_fwi.csv


## Prochaines étapes (hors de ce notebook)

1. Définir la règle de décision paramétrable (seuil relevé si `fwi_class` ∈ {low, very_low, ...})
2. Rejouer la règle sur `df` et calculer TP / FP / FN par config de seuils
3. Tracer le compromis FP supprimés vs feux ratés, + matrice de confusion par classe de risque
4. Synthèse et recommandation